# Prueba técnica de Ciencia de Datos — Conecta+

## Objetivo

Construir una solución reproducible para explorar los datos de Conecta+ y, en etapas posteriores, desarrollar un modelo de ciencia de datos.

En esta primera etapa exploraremos únicamente la estructura de la base SQLite: comprobaremos que el archivo exista, identificaremos sus tablas y revisaremos cuántos registros y qué columnas contiene cada una. No mostraremos datos personales ni analizaremos todavía los archivos JSONL.

> **Nota para Google Colab:** antes de ejecutar estas celdas, los archivos del proyecto deben estar disponibles en el entorno de Colab y la carpeta `datos_conecta/` debe conservarse junto a este notebook. La carga o descompresión se implementará en una etapa posterior.

In [17]:
from pathlib import Path
import sqlite3
from contextlib import closing

In [18]:
db_path = Path("datos_conecta") / "conecta.db"
print(f"Ruta esperada: {db_path}")
print(f"¿Existe el archivo?: {db_path.is_file()}")

if not db_path.is_file():
    raise FileNotFoundError(
        f"No se encontró la base SQLite en: {db_path.resolve()}"
    )

db_uri = db_path.resolve().as_uri() + "?mode=ro"

Ruta esperada: datos_conecta\conecta.db
¿Existe el archivo?: True


In [19]:
with closing(sqlite3.connect(db_uri, uri=True)) as con:
    table_names = [
        row[0]
        for row in con.execute(
            "SELECT name FROM sqlite_master "
            "WHERE type = 'table' AND name NOT LIKE 'sqlite_%' "
            "ORDER BY name"
        )
    ]

print(f"Tablas encontradas ({len(table_names)}):")
for table_name in table_names:
    print(f"- {table_name}")

Tablas encontradas (5):
- clientes
- clientes_a_puntuar
- etiquetas_entrenamiento
- facturas
- suscripciones


In [20]:
with closing(sqlite3.connect(db_uri, uri=True)) as con:
    for table_name in table_names:
        quoted_name = '"' + table_name.replace('"', '""') + '"'
        row_count = con.execute(
            f"SELECT COUNT(*) FROM {quoted_name}"
        ).fetchone()[0]
        columns = [
            row[1]
            for row in con.execute(
                f"PRAGMA table_info({quoted_name})"
            )
        ]

        print(f"\nTabla: {table_name}")
        print(f"Número de registros: {row_count}")
        print(f"Columnas ({len(columns)}): {columns}")


Tabla: clientes
Número de registros: 3000
Columnas (10): ['cliente_id', 'fecha_alta', 'segmento', 'ciudad', 'edad', 'canal_adquisicion', 'plataforma_principal', 'estado_actual', 'contactado_por_retencion', 'fecha_contacto_retencion']

Tabla: clientes_a_puntuar
Número de registros: 1549
Columnas (1): ['cliente_id']

Tabla: etiquetas_entrenamiento
Número de registros: 4758
Columnas (3): ['cliente_id', 'fecha_corte', 'churn_90d']

Tabla: facturas
Número de registros: 34267
Columnas (9): ['factura_id', 'cliente_id', 'suscripcion_id', 'periodo', 'monto', 'fecha_vencimiento', 'fecha_pago', 'metodo_pago', 'fecha_carga']

Tabla: suscripciones
Número de registros: 3362
Columnas (6): ['suscripcion_id', 'cliente_id', 'plan', 'fecha_inicio', 'fecha_fin', 'motivo_cancelacion']


# Auditoría inicial de calidad de datos

Esta sección audita las cinco tablas SQLite sin modificar registros ni mostrar filas completas. Revisaremos esquema, valores nulos, claves duplicadas, rangos, fechas y relaciones. Cuando no exista una PK declarada, distinguiremos las claves candidatas operativas de las restricciones formales del esquema.

In [ ]:
audit_findings = {}

def quoted_identifier(name):
    return '"' + name.replace('"', '""') + '"'

with closing(sqlite3.connect(db_uri, uri=True)) as con:
    audit_findings['columnas_esquema'] = 0
    for table_name in table_names:
        quoted_table = quoted_identifier(table_name)
        info = list(con.execute(f'PRAGMA table_info({quoted_table})'))
        audit_findings['columnas_esquema'] += len(info)
        print(f'\nTabla: {table_name}')
        for _, name, data_type, not_null, _, primary_key in info:
            print(f'  - {name}: tipo={data_type or "(no declarado)"}; permite_nulos={not not_null}; clave_primaria={bool(primary_key)}')

In [ ]:
with closing(sqlite3.connect(db_uri, uri=True)) as con:
    audit_findings['celdas_nulas'] = 0
    for table_name in table_names:
        quoted_table = quoted_identifier(table_name)
        print(f'\nNulos en {table_name}:')
        for _, column_name, *_ in con.execute(f'PRAGMA table_info({quoted_table})'):
            quoted_column = quoted_identifier(column_name)
            count = con.execute(f'SELECT COUNT(*) FROM {quoted_table} WHERE {quoted_column} IS NULL').fetchone()[0]
            audit_findings['celdas_nulas'] += count
            print(f'  - {column_name}: {count}')

In [ ]:
candidate_keys = {
    'clientes': [('cliente_id',)],
    'clientes_a_puntuar': [('cliente_id',)],
    'etiquetas_entrenamiento': [('cliente_id', 'fecha_corte')],
    'facturas': [('factura_id',)],
    'suscripciones': [('suscripcion_id',)],
}
with closing(sqlite3.connect(db_uri, uri=True)) as con:
    audit_findings['grupos_clave_duplicados'] = 0
    audit_findings['filas_clave_duplicadas'] = 0
    for table_name, keys in candidate_keys.items():
        quoted_table = quoted_identifier(table_name)
        for key in keys:
            grouped = ', '.join(quoted_identifier(name) for name in key)
            groups = con.execute(f'SELECT COUNT(*) FROM (SELECT {grouped} FROM {quoted_table} GROUP BY {grouped} HAVING COUNT(*) > 1)').fetchone()[0]
            rows = con.execute(f'SELECT COALESCE(SUM(n), 0) FROM (SELECT COUNT(*) AS n FROM {quoted_table} GROUP BY {grouped} HAVING COUNT(*) > 1)').fetchone()[0]
            audit_findings['grupos_clave_duplicados'] += groups
            audit_findings['filas_clave_duplicadas'] += rows
            print(f'{table_name} ({", ".join(key)}): grupos={groups}; filas={rows}')

In [ ]:
with closing(sqlite3.connect(db_uri, uri=True)) as con:
    checks = {
        'edades_fuera_de_rango': 'SELECT COUNT(*) FROM clientes WHERE edad IS NOT NULL AND (edad < 0 OR edad > 120)',
        'montos_negativos': 'SELECT COUNT(*) FROM facturas WHERE monto IS NOT NULL AND monto < 0',
        'montos_iguales_a_cero': 'SELECT COUNT(*) FROM facturas WHERE monto IS NOT NULL AND monto = 0',
        'contactos_antes_del_alta': 'SELECT COUNT(*) FROM clientes WHERE fecha_contacto_retencion IS NOT NULL AND fecha_alta IS NOT NULL AND date(fecha_contacto_retencion) < date(fecha_alta)',
        'finalizacion_antes_del_inicio': 'SELECT COUNT(*) FROM suscripciones WHERE fecha_fin IS NOT NULL AND fecha_inicio IS NOT NULL AND date(fecha_fin) < date(fecha_inicio)',
    }
    for label, query in checks.items():
        audit_findings[label] = con.execute(query).fetchone()[0]
        print(f'{label}: {audit_findings[label]}')
    date_columns = {'clientes': ['fecha_alta', 'fecha_contacto_retencion'], 'etiquetas_entrenamiento': ['fecha_corte'], 'facturas': ['fecha_vencimiento', 'fecha_pago', 'fecha_carga'], 'suscripciones': ['fecha_inicio', 'fecha_fin']}
    audit_findings['fechas_no_interpretables'] = 0
    for table_name, columns in date_columns.items():
        for column_name in columns:
            qtable = quoted_identifier(table_name); qcolumn = quoted_identifier(column_name)
            count = con.execute(f'SELECT COUNT(*) FROM {qtable} WHERE {qcolumn} IS NOT NULL AND date({qcolumn}) IS NULL').fetchone()[0]
            audit_findings['fechas_no_interpretables'] += count
            print(f'{table_name}.{column_name}: fechas_no_interpretables={count}')
    minimum, maximum = con.execute('SELECT MIN(monto), MAX(monto) FROM facturas WHERE monto IS NOT NULL').fetchone()
    print(f'facturas.monto: minimo={minimum}; maximo={maximum}')

In [ ]:
relationship_checks = {
    'suscripciones_sin_cliente': 'SELECT COUNT(*) FROM suscripciones s LEFT JOIN clientes c ON c.cliente_id = s.cliente_id WHERE c.cliente_id IS NULL',
    'facturas_sin_cliente': 'SELECT COUNT(*) FROM facturas f LEFT JOIN clientes c ON c.cliente_id = f.cliente_id WHERE c.cliente_id IS NULL',
    'facturas_sin_suscripcion': 'SELECT COUNT(*) FROM facturas f LEFT JOIN suscripciones s ON s.suscripcion_id = f.suscripcion_id WHERE s.suscripcion_id IS NULL',
    'facturas_cliente_distinto_de_suscripcion': 'SELECT COUNT(*) FROM facturas f JOIN suscripciones s ON s.suscripcion_id = f.suscripcion_id WHERE f.cliente_id <> s.cliente_id',
    'clientes_a_puntuar_sin_cliente': 'SELECT COUNT(*) FROM clientes_a_puntuar p LEFT JOIN clientes c ON c.cliente_id = p.cliente_id WHERE c.cliente_id IS NULL',
    'etiquetas_sin_cliente': 'SELECT COUNT(*) FROM etiquetas_entrenamiento e LEFT JOIN clientes c ON c.cliente_id = e.cliente_id WHERE c.cliente_id IS NULL',
}
with closing(sqlite3.connect(db_uri, uri=True)) as con:
    for label, query in relationship_checks.items():
        audit_findings[label] = con.execute(query).fetchone()[0]
        print(f'{label}: {audit_findings[label]}')

## Resumen de la auditoría SQLite

Los valores distintos de cero son señales para la etapa de limpieza; esta auditoría no corrige registros.

In [ ]:
print('Resumen cuantitativo SQLite:')
for label, count in audit_findings.items():
    print(f'- {label}: {count}')

# Exploración y normalización de datos no estructurados

Esta sección trabaja sobre copias en memoria de `tickets.jsonl` y `eventos.jsonl.gz`. No se muestran textos personales ni se modifican los archivos fuente. En Colab, usa `%pip install -q pandas matplotlib` si faltan dependencias.

Las fechas se normalizan a UTC cuando es posible. Las fechas sin zona horaria se documentan como un supuesto: no se inventa una zona; se conservan como `sin_zona` y requieren una decisión de negocio. Para variables históricas se filtrará toda observación posterior a la fecha de corte.

In [ ]:
import gzip
import hashlib
import json
import re
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

tickets_path = db_path.parent / 'tickets.jsonl'
events_path = db_path.parent / 'eventos.jsonl.gz'

def load_jsonl(path, compressed=False):
    opener = gzip.open if compressed else open
    with opener(path, 'rt', encoding='utf-8') as file:
        return [json.loads(line) for line in file if line.strip()]

def record_fingerprint(record):
    text = json.dumps(record, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

tickets_raw = load_jsonl(tickets_path)
events_raw = load_jsonl(events_path, compressed=True)
print(f'tickets: {len(tickets_raw):,}')
print(f'eventos: {len(events_raw):,}')

In [ ]:
def field_profile(records):
    rows = []
    total = len(records)
    for field in sorted({key for record in records for key in record}):
        present = [record[field] for record in records if field in record]
        non_null = [value for value in present if value is not None]
        rows.append({'campo': field, 'presentes': len(present), 'nulos': sum(value is None for value in present), 'ausentes': total - len(present), 'tipos': ', '.join(sorted({type(value).__name__ for value in non_null}))})
    return pd.DataFrame(rows)

print(field_profile(tickets_raw).to_string(index=False))
print('Variantes:', dict(Counter('moderno' if 'cliente' in r else 'legado' for r in tickets_raw)))
print('Esquemas distintos:', len({tuple(sorted(r)) for r in tickets_raw}))
ticket_hashes = pd.Series([record_fingerprint(r) for r in tickets_raw])
print('Tickets duplicados exactos adicionales:', int(ticket_hashes.duplicated().sum()))
for field in ['ticket_id', 'id']:
    values = [r.get(field) for r in tickets_raw if r.get(field) is not None]
    print(f'{field}: repetidos={len(values) - len(set(values))}; ausentes={len(tickets_raw) - len(values)}')

In [ ]:
def nested(record, field, key):
    value = record.get(field)
    return value.get(key) if isinstance(value, dict) else None

def normalize_ticket_client_id(record):
    modern_value = nested(record, 'cliente', 'id')
    if modern_value is not None:
        candidate = str(modern_value).strip()
        if re.fullmatch(r'\d+', candidate):
            return int(candidate), 'moderno', None
        return None, 'moderno', 'cliente.id_no_numerico'
    legacy_value = record.get('customer')
    if legacy_value is not None:
        candidate = str(legacy_value).strip()
        match = re.fullmatch(r'CL-(\d+)', candidate)
        if match:
            return int(match.group(1)), 'legado', None
        return None, 'legado', 'customer_formato_invalido'
    return None, 'desconocido', 'identificador_ausente'

def customer_text(record):
    if isinstance(record.get('text'), str):
        return record['text']
    messages = record.get('mensajes', [])
    return ' '.join(item.get('texto', '') for item in messages if isinstance(item, dict) and isinstance(item.get('texto'), str)) or None

ticket_rows = []
for record in tickets_raw:
    cliente_id, schema_variant, id_error = normalize_ticket_client_id(record)
    ticket_rows.append({'ticket_id': record.get('ticket_id', record.get('id')), 'cliente_id': cliente_id, 'cliente_id_error': id_error, 'created_at_raw': record.get('created_at', record.get('fecha')), 'resolved_at_raw': record.get('resolved_at'), 'category': record.get('category', nested(record, 'tipo', 'principal')), 'channel': record.get('channel', record.get('canal')), 'status': record.get('status', nested(record, 'estado', 'valor')), 'satisfaction': record.get('csat', nested(record, 'satisfaccion', 'csat')), 'customer_text': customer_text(record), 'schema_variant': schema_variant})
tickets_df = pd.DataFrame(ticket_rows)
status_map = {'open': 'abierto', 'abierto': 'abierto', 'closed': 'resuelto', 'resuelto': 'resuelto'}
tickets_df['status_normalized'] = tickets_df['status'].astype('string').str.lower().map(status_map)
tickets_df['created_at_utc'] = pd.to_datetime(tickets_df['created_at_raw'], errors='coerce', utc=True, format='mixed')
tickets_df['resolved_at_utc'] = pd.to_datetime(tickets_df['resolved_at_raw'], errors='coerce', utc=True, format='mixed')
tickets_df['created_timezone'] = tickets_df['created_at_raw'].astype('string').str.extract(r'(Z|[+-]\d{2}:?\d{2})$', expand=False).fillna('sin_zona')
print('tickets_df:', tickets_df.shape)
print('Variantes:', tickets_df['schema_variant'].value_counts(dropna=False).to_dict())
print('Identificadores no interpretables por formato:')
print(tickets_df['cliente_id_error'].value_counts(dropna=False).to_string())
print('Nulos por columna:')
print(tickets_df.isna().sum().to_string())
print('Texto presente:', int(tickets_df['customer_text'].notna().sum()), 'de', len(tickets_df))
print('Zonas de fecha de creación:')
print(tickets_df['created_timezone'].value_counts().to_string())
print('Categorías:')
print(tickets_df['category'].value_counts().to_string())
print('Estados:')
print(tickets_df['status'].value_counts().to_string())
print('Estados no mapeados:', int(tickets_df['status_normalized'].isna().sum()))

In [ ]:
events_df = pd.DataFrame(events_raw).copy()
events_df['ts_raw'] = events_df['ts']
events_df['ts_utc'] = pd.to_datetime(events_df['ts'], errors='coerce', utc=True, format='mixed')
events_df['timezone_token'] = events_df['ts_raw'].str.extract(r'(Z|[+-]\d{2}:?\d{2})$', expand=False).fillna('sin_zona')
event_hashes = pd.Series([record_fingerprint(r) for r in events_raw])
print('events_df:', events_df.shape)
print('Nulos por columna:')
print(events_df.isna().sum().to_string())
print('Duplicados exactos adicionales:', int(event_hashes.duplicated().sum()))
print('Fechas no interpretables:', int(events_df['ts_utc'].isna().sum()))
print('Zonas:', events_df['timezone_token'].value_counts().to_string())
print('Tipos:', events_df['tipo'].value_counts().to_string())
print('Plataformas:', events_df['plataforma'].value_counts().to_string())
print('Errores:', events_df['codigo_error'].fillna('sin_error').value_counts().to_string())

## Relación con clientes y prevención de fuga temporal

Los identificadores se comparan con SQLite. El conteo posterior a la fecha de corte se interpreta como pares evento-corte, porque un cliente puede tener varias fechas de corte.

In [ ]:
with closing(sqlite3.connect(db_uri, uri=True)) as con:
    clients = pd.read_sql_query('SELECT cliente_id FROM clientes', con)
    cutoffs = pd.read_sql_query('SELECT cliente_id, fecha_corte FROM etiquetas_entrenamiento', con)
known_ids = set(clients['cliente_id'].astype('int64'))
event_ids = set(events_df['cliente_id'].astype('int64'))
ticket_ids = set(tickets_df['cliente_id'].dropna().astype('int64'))
print('Clientes de eventos ausentes en SQLite:', len(event_ids - known_ids))
print('Clientes de tickets ausentes en SQLite:', len(ticket_ids - known_ids))
cutoffs['fecha_corte_utc'] = pd.to_datetime(cutoffs['fecha_corte'], errors='coerce', utc=True, format='mixed')
pairs = events_df[['cliente_id', 'ts_utc']].merge(cutoffs[['cliente_id', 'fecha_corte_utc']], on='cliente_id', how='inner')
future_pairs = pairs['ts_utc'] > pairs['fecha_corte_utc']
print('Pares evento-corte evaluados:', len(pairs))
historical_event_pairs = pairs.loc[~future_pairs].copy()
assert not (historical_event_pairs['ts_utc'] > historical_event_pairs['fecha_corte_utc']).fillna(False).any()
print('Pares posteriores al corte:', int(future_pairs.sum()))
print('Pares conservados para históricos:', len(historical_event_pairs))
print('Cortes no interpretables:', int(cutoffs['fecha_corte_utc'].isna().sum()))

In [ ]:
ticket_summary = tickets_df.groupby(['schema_variant', 'status'], dropna=False).size().reset_index(name='tickets')
event_summary = events_df.groupby(['tipo', 'plataforma'], dropna=False).size().reset_index(name='eventos')
print(ticket_summary.to_string(index=False))
print(event_summary.to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
tickets_df['status'].value_counts().plot.bar(ax=axes[0], title='Tickets por estado')
events_df['tipo'].value_counts().plot.bar(ax=axes[1], title='Eventos por tipo')
plt.tight_layout()
plt.show()

## Decisiones y pendientes

No se eliminan duplicados. El texto libre se conserva en memoria pero no se muestra. Las fechas sin zona quedan documentadas como `sin_zona`; no se les asigna una zona arbitraria. Antes de construir variables históricas se debe filtrar por `cliente_id` y `fecha_corte`, excluyendo pares posteriores al corte. El entrenamiento del modelo queda fuera de esta etapa.

# Etapa 5 — Integración temporal e ingeniería de características

Construiremos una tabla de entrenamiento con una fila por `(cliente_id, fecha_corte)` y una tabla de predicción con una fila por `cliente_id`. Las fuentes se agregan primero a ese nivel para evitar multiplicaciones de filas.

No se usarán `estado_actual`, `motivo_cancelacion` ni datos posteriores al corte. Las ventanas de actividad, tickets y facturación serán de 7, 30 y 90 días cuando la fuente lo permita.

## 1. Bases temporales y fuentes

Las tres fechas históricas se toman de `etiquetas_entrenamiento`. Para `clientes_a_puntuar` se fija el corte del 31/12/2025. La etiqueta `churn_90d` se conserva solo en entrenamiento y nunca se incluye entre las variables predictoras.

In [ ]:
with closing(sqlite3.connect(db_uri, uri=True)) as con:
    labels = pd.read_sql_query('SELECT cliente_id, fecha_corte, churn_90d FROM etiquetas_entrenamiento', con)
    score_clients = pd.read_sql_query('SELECT cliente_id FROM clientes_a_puntuar', con)
    clients_sql = pd.read_sql_query('SELECT cliente_id, fecha_alta, segmento, edad FROM clientes', con)
    subscriptions_sql = pd.read_sql_query('SELECT suscripcion_id, cliente_id, plan, fecha_inicio, fecha_fin FROM suscripciones', con)
    invoices_sql = pd.read_sql_query('SELECT factura_id, cliente_id, suscripcion_id, periodo, monto, fecha_vencimiento, fecha_pago, fecha_carga FROM facturas', con)

labels['fecha_corte'] = pd.to_datetime(labels['fecha_corte'], utc=True)
score_cutoff = pd.Timestamp('2025-12-31', tz='UTC')
score_base = score_clients.assign(fecha_corte=score_cutoff)
train_base = labels.copy()
train_base['dataset'] = 'train'
score_base['dataset'] = 'predict'
score_base['churn_90d'] = pd.NA
base = pd.concat([train_base, score_base], ignore_index=True, sort=False)
print('Cortes de entrenamiento:', sorted(labels['fecha_corte'].dt.strftime('%Y-%m-%d').unique()))
print('Filas entrenamiento:', len(train_base), '| clientes predicción:', score_base['cliente_id'].nunique())

## 2. Políticas de disponibilidad y duplicados

- Facturas: se ordenan por `factura_id` y `fecha_carga` y se conserva la primera versión observada. Es una decisión conservadora: evita contar dos veces una factura y no incorpora una corrección cargada después.
- Eventos: se conserva una fila por `event_id`, ordenada por `ts_utc`; los duplicados de identificador se tratan como reintentos/copias hasta que negocio confirme lo contrario.
- Tickets: se usan solo identificadores que puedan convertirse a entero y existir en `clientes`; los demás no se fuerzan a una relación.
- La fecha de disponibilidad de facturas es `fecha_carga`; para eventos y tickets es su timestamp de ocurrencia.

In [ ]:
clients_sql['cliente_id'] = pd.to_numeric(clients_sql['cliente_id'], errors='coerce').astype('Int64')
subscriptions_sql['cliente_id'] = pd.to_numeric(subscriptions_sql['cliente_id'], errors='coerce').astype('Int64')
subscriptions_sql['fecha_inicio'] = pd.to_datetime(subscriptions_sql['fecha_inicio'], utc=True, errors='coerce')
subscriptions_sql['fecha_fin'] = pd.to_datetime(subscriptions_sql['fecha_fin'], utc=True, errors='coerce')
invoices_sql['cliente_id'] = pd.to_numeric(invoices_sql['cliente_id'], errors='coerce').astype('Int64')
for column in ['fecha_vencimiento', 'fecha_pago', 'fecha_carga']:
    invoices_sql[column] = pd.to_datetime(invoices_sql[column], utc=True, errors='coerce')

invoice_duplicate_groups = int(invoices_sql['factura_id'].duplicated(False).sum())
invoice_by_id = invoices_sql.sort_values(['factura_id', 'fecha_carga']).drop_duplicates('factura_id', keep='first').copy()
business_key = ['cliente_id', 'suscripcion_id', 'periodo']
business_duplicate_mask = invoice_by_id.duplicated(business_key, False)
invoice_business_duplicate_rows = int(business_duplicate_mask.sum())
invoice_business_duplicate_group_count = int(invoice_by_id.loc[business_duplicate_mask].groupby(business_key, dropna=False).ngroups)
invoice_business_conflict_group_count = int(invoice_by_id.loc[business_duplicate_mask].groupby(business_key, dropna=False).filter(lambda group: group['cliente_id'].nunique() > 1 or group['monto'].nunique() > 1).groupby(business_key, dropna=False).ngroups)
invoice_features_source = invoice_by_id.drop_duplicates(['cliente_id', 'suscripcion_id', 'periodo'], keep='first').copy()
event_features_source = events_df.sort_values(['event_id', 'ts_utc']).drop_duplicates('event_id', keep='first').copy()
tickets_features_source = tickets_df.copy()
known_ids = set(clients_sql['cliente_id'].dropna().astype('int64'))
ticket_id_coverage = int(tickets_features_source['cliente_id'].isin(known_ids).sum())
ticket_client_coverage = int(tickets_features_source.loc[tickets_features_source['cliente_id'].isin(known_ids), 'cliente_id'].nunique())
ticket_invalid_format_count = int(tickets_features_source['cliente_id'].isna().sum())
print('Filas factura involucradas en IDs duplicados:', invoice_duplicate_groups)
print('Grupos duplicados por suscripción y período:', invoice_business_duplicate_group_count)
print('Filas duplicadas por suscripción y período:', invoice_business_duplicate_rows)
print('Grupos con conflicto de cliente o monto:', invoice_business_conflict_group_count)
print('Filas evento tras deduplicar event_id:', len(event_features_source))
print('Tickets con identificador normalizado y conocido:', ticket_id_coverage, 'de', len(tickets_features_source))
print('Clientes distintos vinculados desde tickets:', ticket_client_coverage)
print('Tickets con identificador inválido o ausente:', ticket_invalid_format_count)

## 3. Funciones de agregación temporal

Cada función recibe la base de cortes y devuelve únicamente agregados por `(cliente_id, fecha_corte)`. Los cruces intermedios se filtran por `0 <= días_desde_evento <= ventana`, evitando usar información futura.

In [ ]:
def empty_numeric(base, columns):
    result = base[['cliente_id', 'fecha_corte']].drop_duplicates().copy()
    for column in columns:
        result[column] = 0.0
    return result

def aggregate_events(base, events, windows=(7, 30, 90)):
    source = events[['event_id', 'cliente_id', 'ts_utc', 'tipo', 'funcionalidad', 'codigo_error']].copy()
    source['cliente_id'] = pd.to_numeric(source['cliente_id'], errors='coerce').astype('Int64')
    pairs = base[['cliente_id', 'fecha_corte']].merge(source, on='cliente_id', how='left')
    pairs['days_before'] = (pairs['fecha_corte'] - pairs['ts_utc']).dt.total_seconds() / 86400
    pairs = pairs[pairs['days_before'].between(0, max(windows), inclusive='both')].copy()
    pairs['sesion'] = (pairs['tipo'] == 'login').astype(int)
    pairs['funcionalidad_evento'] = (pairs['tipo'] == 'uso_funcionalidad').astype(int)
    pairs['error_evento'] = (pairs['tipo'] == 'error_app').astype(int)
    output = base[['cliente_id', 'fecha_corte']].drop_duplicates().copy()
    for window in windows:
        subset = pairs[pairs['days_before'] <= window]
        grouped = subset.groupby(['cliente_id', 'fecha_corte'], as_index=False).agg(**{f'eventos_{window}d': ('event_id', 'size'), f'sesiones_{window}d': ('sesion', 'sum'), f'funcionalidades_{window}d': ('funcionalidad_evento', 'sum'), f'errores_{window}d': ('error_evento', 'sum'), f'funcionalidades_distintas_{window}d': ('funcionalidad', 'nunique')})
        output = output.merge(grouped, on=['cliente_id', 'fecha_corte'], how='left')
    numeric = [column for column in output if column not in ['cliente_id', 'fecha_corte']]
    output[numeric] = output[numeric].fillna(0)
    return output

def aggregate_tickets(base, tickets, windows=(7, 30, 90)):
    source = tickets[tickets['cliente_id'].isin(known_ids)].copy()
    source['satisfaction_value'] = pd.to_numeric(source['satisfaction'], errors='coerce')
    keyword_patterns = {
        'tecnico': r'\b(?:error|fall[oa]|falla|bug|crash|no funciona|problema tecnico|lentitud|lento|timeout|conexi[oó]n|app)\b',
        'cobros': r'\b(?:cobro|cargo|factura|pago|pagado|cobrado|duplicado)\b',
        'precios': r'\b(?:precio|costo|coste|tarifa|plan|cambio de plan)\b',
        'cancelacion': r'\b(?:cancelar|cancelaci[oó]n|dar de baja|baja|suscripci[oó]n)\b'
    }
    keyword_columns = []
    for name, pattern in keyword_patterns.items():
        column = f'keyword_{name}'
        source[column] = source['customer_text'].astype('string').str.contains(pattern, case=False, regex=True, na=False).astype(int)
        keyword_columns.append(column)
    source_columns = ['cliente_id', 'created_at_utc', 'resolved_at_utc', 'ticket_id', 'status_normalized', 'satisfaction_value'] + keyword_columns
    pairs = base[['cliente_id', 'fecha_corte']].merge(source[source_columns], on='cliente_id', how='left')
    pairs['days_before'] = (pairs['fecha_corte'] - pairs['created_at_utc']).dt.total_seconds() / 86400
    pairs = pairs[pairs['days_before'].between(0, max(windows), inclusive='both')].copy()
    pairs['resolved_asof'] = pairs['resolved_at_utc'].notna() & (pairs['resolved_at_utc'] <= pairs['fecha_corte'])
    pairs['satisfaction_asof'] = pairs['satisfaction_value'].where(pairs['resolved_asof'])
    pairs['ticket_open_asof'] = pairs['status_normalized'].where(~pairs['resolved_at_utc'].gt(pairs['fecha_corte']), 'abierto').eq('abierto').astype(int)
    output = base[['cliente_id', 'fecha_corte']].drop_duplicates().copy()
    for window in windows:
        subset = pairs[pairs['days_before'] <= window]
        aggregations = {f'tickets_{window}d': ('ticket_id', 'size'), f'tickets_abiertos_{window}d': ('ticket_open_asof', 'sum'), f'csat_promedio_{window}d': ('satisfaction_asof', 'mean'), f'csat_conocido_{window}d': ('satisfaction_asof', lambda values: values.notna().sum())}
        aggregations.update({f'tickets_{name}_{window}d': (column, 'sum') for name, column in zip(keyword_patterns, keyword_columns)})
        grouped = subset.groupby(['cliente_id', 'fecha_corte'], as_index=False).agg(**aggregations)
        output = output.merge(grouped, on=['cliente_id', 'fecha_corte'], how='left')
    numeric = [column for column in output if column not in ['cliente_id', 'fecha_corte']]
    output[numeric] = output[numeric].fillna(0)
    return output

In [ ]:
def aggregate_invoices(base, invoices, windows=(7, 30, 90)):
    pairs = base[['cliente_id', 'fecha_corte']].merge(invoices, on='cliente_id', how='left')
    pairs['days_before'] = (pairs['fecha_corte'] - pairs['fecha_carga']).dt.total_seconds() / 86400
    pairs = pairs[pairs['days_before'].between(0, max(windows), inclusive='both')].copy()
    pairs['paid_asof'] = pairs['fecha_pago'].notna() & (pairs['fecha_pago'] <= pairs['fecha_corte'])
    pairs['overdue_asof'] = (pairs['fecha_vencimiento'] < pairs['fecha_corte']) & ~pairs['paid_asof']
    output = base[['cliente_id', 'fecha_corte']].drop_duplicates().copy()
    for window in windows:
        subset = pairs[pairs['days_before'] <= window]
        grouped = subset.groupby(['cliente_id', 'fecha_corte'], as_index=False).agg(**{f'facturas_{window}d': ('factura_id', 'size'), f'facturacion_{window}d': ('monto', 'sum'), f'pagos_atrasados_{window}d': ('overdue_asof', 'sum'), f'monto_atrasado_{window}d': ('monto', lambda values: values[subset.loc[values.index, 'overdue_asof']].sum())})
        output = output.merge(grouped, on=['cliente_id', 'fecha_corte'], how='left')
    numeric = [column for column in output if column not in ['cliente_id', 'fecha_corte']]
    output[numeric] = output[numeric].fillna(0)
    return output

def customer_and_subscription_features(base, clients, subscriptions):
    output = base[['cliente_id', 'fecha_corte', 'dataset']].drop_duplicates().merge(clients[['cliente_id', 'fecha_alta', 'segmento', 'edad']], on='cliente_id', how='left')
    output['antiguedad_dias'] = (output['fecha_corte'] - pd.to_datetime(output['fecha_alta'], utc=True, errors='coerce')).dt.days
    sub_pairs = base[['cliente_id', 'fecha_corte']].drop_duplicates().merge(subscriptions, on='cliente_id', how='left')
    active = sub_pairs[(sub_pairs['fecha_inicio'] <= sub_pairs['fecha_corte']) & (sub_pairs['fecha_fin'].isna() | (sub_pairs['fecha_fin'] >= sub_pairs['fecha_corte']))].copy()
    active = active.sort_values(['cliente_id', 'fecha_corte', 'fecha_inicio']).drop_duplicates(['cliente_id', 'fecha_corte'], keep='last')
    active = active[['cliente_id', 'fecha_corte', 'plan', 'fecha_inicio']].rename(columns={'plan': 'plan_vigente'})
    output = output.merge(active, on=['cliente_id', 'fecha_corte'], how='left')
    output['plan_vigente'] = output['plan_vigente'].fillna('sin_plan')
    output['dias_plan_vigente'] = (output['fecha_corte'] - output['fecha_inicio']).dt.days.fillna(0)
    return output.drop(columns=['fecha_alta', 'fecha_inicio'])

## 4. Construcción de tablas finales

Se combinan primero los agregados de cada fuente y después se separan entrenamiento y predicción. `churn_90d` queda fuera de las columnas predictoras.

In [ ]:
features = customer_and_subscription_features(base, clients_sql, subscriptions_sql)
features = features.merge(aggregate_invoices(base, invoice_features_source), on=['cliente_id', 'fecha_corte'], how='left')
features = features.merge(aggregate_tickets(base, tickets_features_source), on=['cliente_id', 'fecha_corte'], how='left')
features = features.merge(aggregate_events(base, event_features_source), on=['cliente_id', 'fecha_corte'], how='left')

non_predictor = {'cliente_id', 'fecha_corte', 'dataset', 'churn_90d'}
predictor_columns = [column for column in features.columns if column not in non_predictor]
train_table = features[features['dataset'].eq('train')].merge(labels, on=['cliente_id', 'fecha_corte'], how='left')
prediction_table = features[features['dataset'].eq('predict')].copy()
print('Dimensión entrenamiento:', train_table.shape)
print('Dimensión predicción:', prediction_table.shape)
print('Características predictoras:', len(predictor_columns))
print(predictor_columns)

In [ ]:
assert len(train_table) == 4758
assert train_table[['cliente_id', 'fecha_corte']].duplicated().sum() == 0
assert len(prediction_table) == 1549
assert prediction_table['cliente_id'].duplicated().sum() == 0
assert set(train_table[predictor_columns].columns) == set(prediction_table[predictor_columns].columns)
assert len(predictor_columns) == 56
assert 'churn_90d' not in predictor_columns
assert ticket_id_coverage == len(tickets_features_source) == 4681
assert ticket_client_coverage == tickets_features_source['cliente_id'].nunique()
assert not (historical_event_pairs['ts_utc'] > historical_event_pairs['fecha_corte_utc']).fillna(False).any()
historical_ticket_pairs = train_base[['cliente_id', 'fecha_corte']].merge(tickets_features_source[['cliente_id', 'created_at_utc']], on='cliente_id', how='inner')
historical_ticket_pairs['days_before'] = (historical_ticket_pairs['fecha_corte'] - historical_ticket_pairs['created_at_utc']).dt.total_seconds() / 86400
used_ticket_pairs = historical_ticket_pairs[historical_ticket_pairs['days_before'].between(0, 90, inclusive='both')]
assert not (used_ticket_pairs['created_at_utc'] > used_ticket_pairs['fecha_corte']).fillna(False).any()
invoice_cutoff_pairs = train_base[['cliente_id', 'fecha_corte']].merge(invoice_features_source[['cliente_id', 'fecha_pago']], on='cliente_id', how='inner')
invoice_cutoff_pairs['paid_asof'] = invoice_cutoff_pairs['fecha_pago'].notna() & (invoice_cutoff_pairs['fecha_pago'] <= invoice_cutoff_pairs['fecha_corte'])
assert not (invoice_cutoff_pairs['fecha_pago'] > invoice_cutoff_pairs['fecha_corte']).fillna(False).where(invoice_cutoff_pairs['paid_asof'], False).any()
missing_summary = train_table[predictor_columns].isna().sum().sort_values(ascending=False)
assert missing_summary.sum() == 0
assert not (invoice_features_source['fecha_carga'] > pd.Timestamp('2025-12-31', tz='UTC')).any()
print('Validaciones: OK')
print('Duplicados train:', int(train_table[['cliente_id', 'fecha_corte']].duplicated().sum()))
print('Duplicados prediction:', int(prediction_table['cliente_id'].duplicated().sum()))
print('Tickets vinculados:', ticket_id_coverage, 'de', len(tickets_features_source))
print('Clientes distintos vinculados:', ticket_client_coverage)
print('Valores faltantes por predictor:')
print(missing_summary[missing_summary > 0].to_string())
print('Pares históricos sin información posterior al corte: OK')

## Supuestos y riesgos pendientes

- `fecha_carga` se interpreta como la fecha de disponibilidad de una factura. Los pagos posteriores al corte no se consideran pagados en `paid_asof`; una factura vencida y no pagada a esa fecha se cuenta como atrasada.
- Los tickets se normalizan desde `cliente.id` numérico o `customer` con patrón `CL-<número>`. Los formatos inválidos se reportan y no se fuerzan a una relación.
- Para tickets creados antes del corte pero resueltos después, el estado se considera abierto al corte y la satisfacción no está disponible.
- Las ventanas de tickets siguen siendo de 7, 30 y 90 días y se agregan antes de unirlas a la base cliente-fecha.
- El texto libre no se imprime: se transforma en conteos interpretables por ventana para palabras clave técnicas, cobros, precios y cancelación.
- `csat_promedio_*d` se imputa a cero cuando no hay satisfacción observable, acompañado de `csat_conocido_*d` para distinguir desconocido de un valor numérico.
- Los valores faltantes de conteos, montos y CSAT sin observaciones en la ventana se imputan a cero para representar ausencia de actividad; las variables categóricas usan `sin_plan` y las fechas faltantes se transforman en duraciones cero cuando no hay suscripción activa.
- Los duplicados se deduplican solo para construir features, sin modificar las fuentes.
- La etapa 5 construye las tablas sin entrenar modelos; el modelado se documenta a continuación como etapa 6.

# Etapa 6 — Entrenamiento y validación del modelo de cancelación

Se comparan una regresión logística y `HistGradientBoostingClassifier` usando una validación temporal. Marzo de 2025 se utiliza para desarrollo, junio para comparación y septiembre queda reservado para la evaluación final. El modelo definitivo se reentrena con los tres cortes históricos únicamente después de completar esa evaluación, y genera probabilidades en memoria para los 1.549 clientes del corte 31/12/2025. No se crea todavía `predicciones.csv` ni se decide a quién contactar.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, confusion_matrix, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
target_column = 'churn_90d'
cutoff_column = 'fecha_corte'
model_data = train_table.copy()
model_data[cutoff_column] = pd.to_datetime(model_data[cutoff_column], utc=True)
assert target_column not in predictor_columns
assert set(predictor_columns).issubset(model_data.columns)

march_cutoff = pd.Timestamp('2025-03-31', tz='UTC')
june_cutoff = pd.Timestamp('2025-06-30', tz='UTC')
september_cutoff = pd.Timestamp('2025-09-30', tz='UTC')
development_train = model_data[model_data[cutoff_column].eq(march_cutoff)].copy()
development_valid = model_data[model_data[cutoff_column].eq(june_cutoff)].copy()
final_eval = model_data[model_data[cutoff_column].eq(september_cutoff)].copy()
X_march, y_march = development_train[predictor_columns], development_train[target_column]
X_june, y_june = development_valid[predictor_columns], development_valid[target_column]
X_september, y_september = final_eval[predictor_columns], final_eval[target_column]
categorical_features = ['segmento', 'plan_vigente']
numeric_features = [column for column in predictor_columns if column not in categorical_features]

def make_preprocessor():
    try:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
    return ColumnTransformer([
        ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', encoder)]), categorical_features)
    ], remainder='drop')

def make_model(name):
    if name == 'logistic':
        estimator = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED)
    elif name == 'hist_gradient_boosting':
        estimator = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15, l2_regularization=1.0, random_state=SEED)
    else:
        raise ValueError(f'Modelo no soportado: {name}')
    return Pipeline([('preprocess', make_preprocessor()), ('model', estimator)])

print('Filas marzo/junio/septiembre:', len(development_train), len(development_valid), len(final_eval))
print('Prevalencia marzo/junio/septiembre:', y_march.mean(), y_june.mean(), y_september.mean())

## Comparación temporal durante el desarrollo

La comparación se realiza entrenando con marzo y evaluando con junio. El `DummyClassifier` representa una referencia que solo conoce la prevalencia de marzo. La selección prioriza PR-AUC, porque es informativa cuando la clase positiva no es necesariamente equilibrada; ROC-AUC y Brier Score se reportan como métricas complementarias.

In [ ]:
def evaluate_probabilities(name, y_true, probabilities, threshold=0.5):
    predictions = (probabilities >= threshold).astype(int)
    return {
        'modelo': name,
        'roc_auc': roc_auc_score(y_true, probabilities),
        'pr_auc': average_precision_score(y_true, probabilities),
        'brier_score': brier_score_loss(y_true, probabilities),
        'umbral': threshold,
        'tn': int(confusion_matrix(y_true, predictions, labels=[0, 1])[0, 0]),
        'fp': int(confusion_matrix(y_true, predictions, labels=[0, 1])[0, 1]),
        'fn': int(confusion_matrix(y_true, predictions, labels=[0, 1])[1, 0]),
        'tp': int(confusion_matrix(y_true, predictions, labels=[0, 1])[1, 1])
    }

development_models = {
    'logistic': make_model('logistic'),
    'hist_gradient_boosting': make_model('hist_gradient_boosting')
}
development_rows = []
development_probabilities = {}
for name, estimator in development_models.items():
    estimator.fit(X_march, y_march)
    probabilities = estimator.predict_proba(X_june)[:, 1]
    development_probabilities[name] = probabilities
    development_rows.append(evaluate_probabilities(name, y_june, probabilities))

dummy = DummyClassifier(strategy='prior').fit(X_march, y_march)
dummy_probabilities = dummy.predict_proba(X_june)[:, 1]
development_rows.append(evaluate_probabilities('dummy_prior', y_june, dummy_probabilities))
development_metrics = pd.DataFrame(development_rows).sort_values(['pr_auc', 'roc_auc'], ascending=False)
candidate_metrics = development_metrics[development_metrics['modelo'].isin(development_models)].copy()
selected_name = candidate_metrics.iloc[0]['modelo']
print(development_metrics.to_string(index=False))
print('Modelo seleccionado para la evaluación final:', selected_name)

## Evaluación final, calibración e interpretación

El modelo seleccionado se ajusta ahora con marzo y junio, sin utilizar septiembre, y se evalúa una sola vez sobre septiembre. El umbral 0.5 se muestra por ser una referencia transparente, no una decisión económica. La curva de calibración y el Brier Score permiten revisar si las probabilidades son razonables; la importancia por permutación se calcula sobre variables originales, no sobre texto personal.

In [ ]:
X_development = pd.concat([X_march, X_june], axis=0)
y_development = pd.concat([y_march, y_june], axis=0)
selected_model = make_model(selected_name)
selected_model.fit(X_development, y_development)
september_probabilities = selected_model.predict_proba(X_september)[:, 1]
final_evaluation = evaluate_probabilities(selected_name, y_september, september_probabilities, threshold=0.5)
dummy_final = DummyClassifier(strategy='prior').fit(X_development, y_development)
dummy_final_probabilities = dummy_final.predict_proba(X_september)[:, 1]
final_comparison = pd.DataFrame([final_evaluation, evaluate_probabilities('dummy_prior', y_september, dummy_final_probabilities, threshold=0.5)])
print('Evaluación reservada de septiembre:')
print(final_comparison.to_string(index=False))

from sklearn.calibration import calibration_curve
fraction_positive, mean_probability = calibration_curve(y_september, september_probabilities, n_bins=10, strategy='quantile')
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(mean_probability, fraction_positive, marker='o', label=selected_name)
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='calibración perfecta')
ax.set_xlabel('Probabilidad media predicha')
ax.set_ylabel('Frecuencia positiva observada')
ax.set_title('Calibración en septiembre')
ax.legend()
plt.tight_layout()
plt.show()

from sklearn.inspection import permutation_importance
permutation = permutation_importance(selected_model, X_september, y_september, scoring='average_precision', n_repeats=3, random_state=SEED, n_jobs=-1)
feature_importance = pd.DataFrame({'variable': predictor_columns, 'importance_mean': permutation.importances_mean, 'importance_std': permutation.importances_std}).sort_values('importance_mean', ascending=False)
print('Variables más importantes por permutación:')
print(feature_importance.head(15).to_string(index=False))

## Ajuste definitivo y probabilidades para diciembre

Una vez registrada la evaluación de septiembre, se reentrena el procedimiento seleccionado con marzo, junio y septiembre. Las probabilidades se generan para `prediction_table` al corte del 31/12/2025 y quedan en memoria; todavía no se escribe `predicciones.csv` ni se establece un umbral de contacto.

In [ ]:
final_model = make_model(selected_name)
final_model.fit(model_data[predictor_columns], model_data[target_column])
prediction_probabilities = pd.DataFrame({
    'cliente_id': prediction_table['cliente_id'].astype('int64'),
    'probabilidad_churn_90d': final_model.predict_proba(prediction_table[predictor_columns])[:, 1]
})
assert len(prediction_probabilities) == 1549
assert prediction_probabilities['cliente_id'].duplicated().sum() == 0
assert prediction_probabilities['probabilidad_churn_90d'].between(0, 1).all()
print('Modelo definitivo:', selected_name)
print('Probabilidades generadas:', len(prediction_probabilities))
print(prediction_probabilities['probabilidad_churn_90d'].describe(percentiles=[0.1, 0.5, 0.9]).to_string())
print('Muestra no sensible de probabilidades:')
print(prediction_probabilities.head(10).to_string(index=False))

# Etapa 7 — Revisión de calibración y fiabilidad

La regresión logística anterior utilizó `class_weight='balanced'`. Esa opción puede mejorar la detección de la clase positiva, pero cambia la escala de las probabilidades porque pondera artificialmente la función de pérdida; por eso se comparan versiones balanceada y no balanceada. La calibración se ajusta solo con información anterior a septiembre. Como septiembre ya fue observado en la etapa 6, cualquier decisión tomada después de ver sus resultados tiene riesgo de selección adaptativa y sus métricas se tratarán como referencia, no como una evaluación completamente nueva.

In [ ]:
import numpy
from sklearn.isotonic import IsotonicRegression

def make_logistic_variant(weighted):
    class_weight = 'balanced' if weighted else None
    estimator = LogisticRegression(max_iter=2000, class_weight=class_weight, random_state=SEED)
    return Pipeline([('preprocess', make_preprocessor()), ('model', estimator)])

def fit_sigmoid_calibrator(probabilities, target):
    clipped = numpy.clip(probabilities, 1e-6, 1 - 1e-6)
    logit = numpy.log(clipped / (1 - clipped)).reshape(-1, 1)
    calibrator = LogisticRegression(C=1e6, max_iter=1000, random_state=SEED)
    calibrator.fit(logit, target)
    return calibrator

def apply_sigmoid_calibrator(calibrator, probabilities):
    clipped = numpy.clip(probabilities, 1e-6, 1 - 1e-6)
    logit = numpy.log(clipped / (1 - clipped)).reshape(-1, 1)
    return calibrator.predict_proba(logit)[:, 1]

def fit_isotonic_calibrator(probabilities, target):
    return IsotonicRegression(out_of_bounds='clip').fit(probabilities, target)

def apply_calibrator(calibrator, method, probabilities):
    if method == 'raw':
        return probabilities
    if method == 'sigmoid':
        return apply_sigmoid_calibrator(calibrator, probabilities)
    return calibrator.predict(probabilities)

# La partición de junio es determinista y separa clientes, evitando que un cliente
# aparezca simultáneamente en el ajuste y en la comparación del calibrador.
calibration_mask = development_valid['cliente_id'].mod(5).ne(0)
calibration_rows = development_valid.loc[calibration_mask]
calibration_holdout_rows = development_valid.loc[~calibration_mask]
calibration_ids = set(calibration_rows['cliente_id'])
calibration_holdout_ids = set(calibration_holdout_rows['cliente_id'])
assert calibration_ids.isdisjoint(calibration_holdout_ids)
X_june_calibration = calibration_rows[predictor_columns]
y_june_calibration = calibration_rows[target_column]
X_june_holdout = calibration_holdout_rows[predictor_columns]
y_june_holdout = calibration_holdout_rows[target_column]

calibration_rows_output = []
calibration_artifacts = {}
for weighted, name in [(True, 'logistic_balanced'), (False, 'logistic_unweighted')]:
    base_model = make_logistic_variant(weighted)
    base_model.fit(X_march, y_march)
    june_calibration_probabilities = base_model.predict_proba(X_june_calibration)[:, 1]
    june_holdout_probabilities = base_model.predict_proba(X_june_holdout)[:, 1]
    calibration_artifacts[name] = {'base_model': base_model, 'holdout_probabilities': june_holdout_probabilities}
    calibration_rows_output.append(evaluate_probabilities(name + '_raw', y_june_holdout, june_holdout_probabilities))
    for method in ['sigmoid', 'isotonic']:
        calibrator = fit_sigmoid_calibrator(june_calibration_probabilities, y_june_calibration) if method == 'sigmoid' else fit_isotonic_calibrator(june_calibration_probabilities, y_june_calibration)
        calibrated_probabilities = apply_calibrator(calibrator, method, june_holdout_probabilities)
        calibration_artifacts[name][method] = calibrator
        calibration_rows_output.append(evaluate_probabilities(name + '_' + method, y_june_holdout, calibrated_probabilities))

calibration_metrics = pd.DataFrame(calibration_rows_output)
raw_pr_auc_reference = calibration_metrics.loc[calibration_metrics['modelo'].str.endswith('_raw'), 'pr_auc'].max()
calibrated_candidates = calibration_metrics[~calibration_metrics['modelo'].str.endswith('_raw')].copy()
acceptable_candidates = calibrated_candidates[calibrated_candidates['pr_auc'] >= 0.95 * raw_pr_auc_reference]
if acceptable_candidates.empty:
    selected_calibrated_row = calibrated_candidates.sort_values(['brier_score', 'pr_auc'], ascending=[True, False]).iloc[0]
else:
    selected_calibrated_row = acceptable_candidates.sort_values(['brier_score', 'pr_auc'], ascending=[True, False]).iloc[0]
selected_reviewed_model = selected_calibrated_row['modelo'].rsplit('_', 1)[0]
selected_calibration_method = selected_calibrated_row['modelo'].rsplit('_', 1)[1]
print('Filas junio para calibrar y comparar:', len(calibration_rows), len(calibration_holdout_rows))
print(calibration_metrics.sort_values(['brier_score', 'pr_auc'], ascending=[True, False]).to_string(index=False))
print('Procedimiento elegido con junio:', selected_reviewed_model, '+', selected_calibration_method)

## Evaluación temporal de la calibración

Los calibradores se ajustaron con predicciones de junio producidas por modelos entrenados solo con marzo. La comparación de junio usa clientes disjuntos. Para septiembre se entrena la base con marzo y junio, pero se conserva el calibrador elegido sin usar etiquetas de septiembre. La comparación de septiembre es informativa, aunque está condicionada por que septiembre ya se había observado en la etapa anterior.

In [ ]:
selected_weighted = selected_reviewed_model.endswith('balanced')
selected_base_for_eval = make_logistic_variant(selected_weighted)
selected_base_for_eval.fit(X_development, y_development)
september_raw_probabilities = selected_base_for_eval.predict_proba(X_september)[:, 1]
selected_calibrator = calibration_artifacts[selected_reviewed_model][selected_calibration_method]
september_reviewed_probabilities = apply_calibrator(selected_calibrator, selected_calibration_method, september_raw_probabilities)
previous_balanced_model = make_logistic_variant(True)
previous_balanced_model.fit(X_development, y_development)
previous_balanced_probabilities = previous_balanced_model.predict_proba(X_september)[:, 1]

september_calibration_comparison = pd.DataFrame([
    evaluate_probabilities('previous_logistic_balanced_raw', y_september, previous_balanced_probabilities),
    evaluate_probabilities(selected_reviewed_model + '_' + selected_calibration_method, y_september, september_reviewed_probabilities),
    evaluate_probabilities('dummy_prior', y_september, dummy_final_probabilities)
])
print('Tasa observada septiembre:', float(y_september.mean()))
print(september_calibration_comparison.to_string(index=False))

from sklearn.calibration import calibration_curve
fraction_raw, mean_raw = calibration_curve(y_september, previous_balanced_probabilities, n_bins=10, strategy='quantile')
fraction_reviewed, mean_reviewed = calibration_curve(y_september, september_reviewed_probabilities, n_bins=10, strategy='quantile')
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(mean_raw, fraction_raw, marker='o', label='logística balanceada')
ax.plot(mean_reviewed, fraction_reviewed, marker='o', label='procedimiento revisado')
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='calibración perfecta')
ax.set_xlabel('Probabilidad media predicha')
ax.set_ylabel('Frecuencia positiva observada')
ax.set_title('Calibración en septiembre')
ax.legend()
plt.tight_layout()
plt.show()

print('Media septiembre raw:', float(previous_balanced_probabilities.mean()))
print('Media septiembre revisada:', float(september_reviewed_probabilities.mean()))

## Probabilidades definitivas para diciembre

Después de la revisión, la base seleccionada se reentrena con los tres cortes históricos. El calibrador se mantiene como el ajustado antes de septiembre; no se aprovechan las etiquetas de septiembre para cambiarlo. Las probabilidades quedan en memoria y no se genera todavía ningún archivo de entrega.

In [ ]:
reviewed_final_base = make_logistic_variant(selected_weighted)
reviewed_final_base.fit(model_data[predictor_columns], model_data[target_column])
december_raw_probabilities = reviewed_final_base.predict_proba(prediction_table[predictor_columns])[:, 1]
december_reviewed_probabilities = apply_calibrator(selected_calibrator, selected_calibration_method, december_raw_probabilities)
reviewed_prediction_probabilities = pd.DataFrame({
    'cliente_id': prediction_table['cliente_id'].astype('int64'),
    'probabilidad_churn_90d': december_reviewed_probabilities
})
assert len(reviewed_prediction_probabilities) == 1549
assert reviewed_prediction_probabilities['cliente_id'].duplicated().sum() == 0
assert reviewed_prediction_probabilities['probabilidad_churn_90d'].between(0, 1).all()
print('Procedimiento definitivo:', selected_reviewed_model, '+', selected_calibration_method)
print('Tasa observada septiembre:', float(y_september.mean()))
print(reviewed_prediction_probabilities['probabilidad_churn_90d'].describe(percentiles=[0.1, 0.5, 0.9]).to_string())
print('Muestra no sensible:')
print(reviewed_prediction_probabilities.head(10).to_string(index=False))
print('No se genera predicciones.csv en esta etapa.')

# Etapa 8 — Decisión económica y generación de predicciones

Se utilizarán las probabilidades calibradas para decidir contactos bajo los supuestos económicos del enunciado. Antes de calcular el valor esperado se corrige la compatibilidad entre el modelo base definitivo y el calibrador: se generan predicciones fuera de muestra con una secuencia temporal expansiva —marzo→junio y marzo+junio→septiembre— y se ajusta nuevamente el calibrador con esas predicciones. Septiembre pasa a formar parte de la calibración para el corte de diciembre, por lo que se conserva la advertencia de selección adaptativa respecto a la evaluación realizada en la etapa 7.

In [ ]:
from contextlib import closing
import sqlite3

COSTO_CONTACTO = 25000
EFECTIVIDAD_RETENCION = 0.40
MESES_RETENIDOS = 12
PLAN_PRECIOS = {'Basico': 23900, 'Estandar': 34900, 'Premium': 49900, 'Familiar': 59900}

stage8_weighted = selected_reviewed_model == 'logistic_balanced'
stage8_method = selected_calibration_method

march_base_for_oof = make_logistic_variant(stage8_weighted)
march_base_for_oof.fit(X_march, y_march)
june_oof_probabilities = march_base_for_oof.predict_proba(X_june)[:, 1]
march_june_base_for_oof = make_logistic_variant(stage8_weighted)
march_june_base_for_oof.fit(X_development, y_development)
september_oof_probabilities = march_june_base_for_oof.predict_proba(X_september)[:, 1]
calibration_oof_probabilities = numpy.concatenate([june_oof_probabilities, september_oof_probabilities])
calibration_oof_target = numpy.concatenate([y_june.to_numpy(), y_september.to_numpy()])
if stage8_method == 'sigmoid':
    stage8_calibrator = fit_sigmoid_calibrator(calibration_oof_probabilities, calibration_oof_target)
else:
    stage8_calibrator = fit_isotonic_calibrator(calibration_oof_probabilities, calibration_oof_target)
stage8_final_base = make_logistic_variant(stage8_weighted)
stage8_final_base.fit(model_data[predictor_columns], model_data[target_column])
stage8_raw_probabilities = stage8_final_base.predict_proba(prediction_table[predictor_columns])[:, 1]
stage8_calibrated_probabilities = apply_calibrator(stage8_calibrator, stage8_method, stage8_raw_probabilities)
print('Predicciones OOF usadas para calibración:', len(calibration_oof_probabilities))
print('Base final compatible:', 'logística_balanceada' if stage8_weighted else 'logística_no_balanceada')
print('Calibrador temporal:', stage8_method)
print('Media probabilidades etapa 7:', float(reviewed_prediction_probabilities['probabilidad_churn_90d'].mean()))
print('Media probabilidades etapa 8:', float(stage8_calibrated_probabilities.mean()))

## Valor esperado y archivo de entrega

Para cada cliente se toma el plan vigente al 31/12/2025 y se aplica `VE = probabilidad × 0.40 × (12 × mensualidad) − 25000`. La decisión es `contactar = 1` únicamente cuando `VE > 0`. El CSV contiene solo las tres columnas exigidas; los demás cálculos permanecen en memoria para auditoría.

In [ ]:
economic_table = prediction_table[['cliente_id', 'plan_vigente']].copy()
economic_table['probabilidad'] = stage8_calibrated_probabilities
economic_table['mensualidad'] = economic_table['plan_vigente'].map(PLAN_PRECIOS)
assert economic_table['mensualidad'].notna().all()
economic_table['valor_bruto_esperado'] = economic_table['probabilidad'] * EFECTIVIDAD_RETENCION * (MESES_RETENIDOS * economic_table['mensualidad'])
economic_table['valor_neto_esperado'] = economic_table['valor_bruto_esperado'] - COSTO_CONTACTO
economic_table['contactar'] = economic_table['valor_neto_esperado'].gt(0).astype(int)
predicciones = economic_table[['cliente_id', 'probabilidad', 'contactar']].copy()
predicciones['cliente_id'] = predicciones['cliente_id'].astype('int64')
predicciones['contactar'] = predicciones['contactar'].astype('int64')
predicciones.to_csv('predicciones.csv', index=False)
with closing(sqlite3.connect(db_uri, uri=True)) as con:
    scoring_ids = pd.read_sql_query('SELECT cliente_id FROM clientes_a_puntuar', con)['cliente_id'].astype('int64')
assert len(predicciones) == 1549
assert set(predicciones['cliente_id']) == set(scoring_ids)
assert predicciones['cliente_id'].duplicated().sum() == 0
assert predicciones['probabilidad'].between(0, 1).all()
assert set(predicciones['contactar'].unique()).issubset({0, 1})
assert predicciones.columns.tolist() == ['cliente_id', 'probabilidad', 'contactar']
contacted = economic_table['contactar'].eq(1)
campaign_cost = float(contacted.sum() * COSTO_CONTACTO)
expected_avoided = float((economic_table.loc[contacted, 'probabilidad'] * EFECTIVIDAD_RETENCION).sum())
expected_gross = float(economic_table.loc[contacted, 'valor_bruto_esperado'].sum())
expected_net = expected_gross - campaign_cost
all_gross = float(economic_table['valor_bruto_esperado'].sum())
all_cost = float(len(economic_table) * COSTO_CONTACTO)
all_net = all_gross - all_cost
print('Clientes a contactar:', int(contacted.sum()), f'({100 * contacted.mean():.2f}%)')
print('Costo total campaña:', campaign_cost, 'COP')
print('Cancelaciones evitadas esperadas:', expected_avoided)
print('Valor bruto esperado:', expected_gross, 'COP')
print('Beneficio neto esperado:', expected_net, 'COP')
print('Escenario contactar a todos — bruto/neto:', all_gross, all_net)
print('Escenario no contactar a nadie — bruto/neto:', 0.0, 0.0)
print('Contactos por plan:')
print(economic_table.groupby('plan_vigente', as_index=False)['contactar'].agg(clientes='size', contactos='sum').to_string(index=False))
print('CSV generado:', Path('predicciones.csv').resolve())

## Sensibilidad económica y limitaciones

La sensibilidad recalcula la decisión con efectividades de 20 %, 40 % y 60 %, manteniendo las mismas probabilidades y costos. El valor esperado es una regla económica bajo supuestos, no una estimación causal: la probabilidad predice cancelación, pero no demuestra que contactar al cliente evite esa cancelación.

In [ ]:
sensitivity_rows = []
for effectiveness in [0.20, 0.40, 0.60]:
    value = economic_table['probabilidad'] * effectiveness * (MESES_RETENIDOS * economic_table['mensualidad'])
    decision = value - COSTO_CONTACTO > 0
    gross = float(value.loc[decision].sum())
    cost = float(decision.sum() * COSTO_CONTACTO)
    sensitivity_rows.append({'efectividad': effectiveness, 'contactos': int(decision.sum()), 'porcentaje': float(100 * decision.mean()), 'costo': cost, 'valor_bruto': gross, 'beneficio_neto': gross - cost, 'cancelaciones_evitadas_esperadas': float((economic_table.loc[decision, 'probabilidad'] * effectiveness).sum())})
sensitivity = pd.DataFrame(sensitivity_rows)
print(sensitivity.to_string(index=False))
print('Archivo generado exclusivamente con columnas:', predicciones.columns.tolist())

# Etapa 9 — Hallazgo senior y operación

Esta etapa documenta una señal de negocio reproducible y un plan de producción. El análisis de eventos usa únicamente eventos con `ts_utc <= fecha_corte`; se deduplica por `event_id` como en las etapas anteriores. No se muestran identificadores, textos ni registros individuales.

In [ ]:
# Hallazgo: exposición previa a la versión 5.3.0 y CRASH_ON_START.
# Se trabaja a nivel (cliente_id, fecha_corte), sin imprimir identificadores.
stage9_events = events_df[['event_id', 'cliente_id', 'ts_utc', 'plataforma', 'version_app', 'codigo_error']].copy()
stage9_events['ts_utc'] = pd.to_datetime(stage9_events['ts_utc'], errors='coerce', utc=True, format='mixed')
stage9_events = stage9_events.sort_values(['event_id', 'ts_utc']).drop_duplicates('event_id', keep='first')
stage9_events['evento_530'] = stage9_events['version_app'].eq('5.3.0')
stage9_events['crash_530'] = stage9_events['evento_530'] & stage9_events['codigo_error'].eq('CRASH_ON_START')

stage9_labels = labels[['cliente_id', 'fecha_corte', 'churn_90d']].copy()
stage9_labels['fecha_corte_utc'] = pd.to_datetime(stage9_labels['fecha_corte'], errors='coerce', utc=True, format='mixed')
stage9_pairs = stage9_labels.merge(stage9_events, on='cliente_id', how='left')
stage9_pairs['pre_corte'] = stage9_pairs['ts_utc'].le(stage9_pairs['fecha_corte_utc'])
stage9_pre = stage9_pairs.loc[stage9_pairs['pre_corte']].copy()
stage9_agg = stage9_pre.groupby(['cliente_id', 'fecha_corte'], as_index=False).agg(
    expuesto_530=('evento_530', 'any'),
    expuesto_crash_530=('crash_530', 'any'),
    eventos_previos=('event_id', 'size'),
    eventos_moviles=('plataforma', lambda s: s.isin(['android', 'ios']).sum()),
)
stage9 = stage9_labels.merge(stage9_agg, on=['cliente_id', 'fecha_corte'], how='left')
for column in ['expuesto_530', 'expuesto_crash_530']:
    stage9[column] = stage9[column].fillna(False)
for column in ['eventos_previos', 'eventos_moviles']:
    stage9[column] = stage9[column].fillna(0)

def stage9_rate(frame):
    return pd.Series({
        'observaciones': len(frame),
        'cancelaciones': int(frame['churn_90d'].sum()),
        'tasa_cancelacion_pct': 100 * frame['churn_90d'].mean(),
    })

stage9_summary = stage9.groupby('expuesto_530').apply(stage9_rate, include_groups=False).reset_index()
stage9_crash_summary = stage9.groupby('expuesto_crash_530').apply(stage9_rate, include_groups=False).reset_index()
stage9_summary['tasa_cancelacion_pct'] = stage9_summary['tasa_cancelacion_pct'].round(4)
stage9_crash_summary['tasa_cancelacion_pct'] = stage9_crash_summary['tasa_cancelacion_pct'].round(4)
stage9_exposed = stage9.loc[stage9['expuesto_530']]
stage9_unexposed = stage9.loc[stage9['expuesto_530'].eq(False)]
stage9_rate_exposed = stage9_exposed['churn_90d'].mean()
stage9_rate_unexposed = stage9_unexposed['churn_90d'].mean()
stage9_rate_difference_pp = 100 * (stage9_rate_exposed - stage9_rate_unexposed)
stage9_rate_ratio = stage9_rate_exposed / stage9_rate_unexposed
stage9_by_cutoff = stage9.groupby(['fecha_corte', 'expuesto_530'], as_index=False).agg(
    observaciones=('churn_90d', 'size'),
    cancelaciones=('churn_90d', 'sum'),
    tasa_cancelacion_pct=('churn_90d', lambda s: 100 * s.mean()),
)
stage9_by_cutoff['tasa_cancelacion_pct'] = stage9_by_cutoff['tasa_cancelacion_pct'].round(4)
stage9_mobile_only = stage9.loc[stage9['eventos_moviles'] > 0]
stage9_mobile_comparison = stage9_mobile_only.groupby('expuesto_530').apply(stage9_rate, include_groups=False).reset_index()
stage9_mobile_comparison['tasa_cancelacion_pct'] = stage9_mobile_comparison['tasa_cancelacion_pct'].round(4)

assert not stage9_pre['ts_utc'].gt(stage9_pre['fecha_corte_utc']).fillna(False).any()
assert len(stage9) == len(labels) == 4758
assert stage9[['cliente_id', 'fecha_corte']].duplicated().sum() == 0
print('Eventos únicos analizados:', len(stage9_events))
print('Eventos de versión 5.3.0:', int(stage9_events['evento_530'].sum()))
print('Eventos 5.3.0 con CRASH_ON_START:', int(stage9_events['crash_530'].sum()))
print('Comparación general:')
print(stage9_summary.to_string(index=False))
print('Comparación específica de CRASH_ON_START en 5.3.0:')
print(stage9_crash_summary.to_string(index=False))
print('Comparación por fecha de corte:')
print(stage9_by_cutoff.to_string(index=False))
print('Solo observaciones con actividad móvil previa:')
print(stage9_mobile_comparison.to_string(index=False))
print(f'Diferencia exposición 5.3.0: {stage9_rate_difference_pp:.4f} puntos porcentuales')
print(f'Razón de tasas descriptiva: {stage9_rate_ratio:.4f}')
print('Pares posteriores al corte excluidos:', int(stage9_pairs['ts_utc'].gt(stage9_pairs['fecha_corte_utc']).fillna(False).sum()))

## Interpretación y recomendación del hallazgo

La exposición previa a 5.3.0 presenta una tasa de cancelación descriptivamente mayor que la no exposición. En los datos, la exposición está concentrada en eventos móviles y aparece en el corte de septiembre, mientras que los clientes expuestos también tienen más actividad previa. Por ello pueden influir la fecha de lanzamiento, la plataforma, el nivel de uso, el segmento y otros cambios simultáneos. La asociación no demuestra que la versión cause cancelación.

Recomendación verificable: priorizar una revisión técnica de 5.3.0 y `CRASH_ON_START`, y medir el efecto de un hotfix mediante un despliegue escalonado o una comparación preespecificada entre cohortes elegibles. Registrar como métricas principales la tasa de crash, actividad posterior y cancelación a 90 días; comparar también por plataforma y nivel de actividad. Si se prueba una campaña de retención, asignar aleatoriamente una parte de los clientes elegibles a un grupo de control y medir cancelaciones y beneficio neto, porque el modelo predictivo no estima por sí solo el efecto causal del contacto.

## Plan de producción

En una ejecución periódica, un proceso versionado cargaría de forma privada la base SQLite y los JSONL, validaría esquemas, identificadores, duplicados, fechas de disponibilidad y cobertura, y detendría la ejecución ante cambios críticos. Después actualizaría las agregaciones por cliente y fecha de corte con las ventanas temporales documentadas, sin usar información posterior al corte. Para clientes activos se construiría la fila del nuevo corte, se aplicaría el pipeline y el calibrador aprobados y se guardarían las probabilidades junto con la versión del código, fecha de ejecución y controles de calidad. La frecuencia inicial puede ser mensual, con seguimiento semanal de tasas de error y de contacto; el reentrenamiento debe revisarse al menos trimestralmente o antes si cambia la distribución, la calibración o la calidad de los datos.

El monitoreo debe incluir filas esperadas, duplicados, nulos, cobertura de identificadores, fechas futuras, distribución de variables, ROC-AUC/PR-AUC cuando lleguen etiquetas, Brier Score, curvas de calibración y tasa real de cancelación. En negocio se deben seguir costo por contacto, cancelaciones evitadas y beneficio neto, comparados con un control. Se debe desconfiar del modelo si faltan fuentes, cambian las definiciones de estados o planes, aparecen probabilidades extremas, cae la calibración, hay deriva de población o la intervención no tiene evidencia causal. La efectividad real de retención se medirá con un experimento controlado y preespecificado, no con la diferencia observacional entre contactados y no contactados.